In [2]:
SYSTEM_PROMPT = """\
Твоя задача — проанализировать отзывы клиентов Газпромбанка (ГПБ). Для каждого отзыва тебе необходимо:
1.  Определить все продукты/услуги (темы), о которых упоминает клиент.
2.  Для каждой упомянутой темы определить тональность высказывания: positive, negative или neutral.

**ВАЖНЫЕ ИНСТРУКЦИИ:**
-   **Темы:** В основном используй следующий список: Дебетовые карты, Офисное обслуживание, Дистанционное обслуживание, Кредиты наличными, Кредитные карты, Вклады, Ипотека, Автокредиты, Реструктуризация кредитов, Рефинансирование кредитов, Рефинансирование ипотеки, Обмен валют, Мобильное приложение, Потребительский кредит. Если в отзыве описана другая тема, ты можешь добавить ее. Помимо этого, можешь указывать уточняющие темы, такие как продукты банка, которые приведены ниже.
-   **Тональность:** `neutral` указывается тогда, когда тема упомянута как факт, без какой-либо эмоциональной окраски (ни положительной, ни отрицательной).
-   **Строгость:** Не выдумывай темы. Если в отзыве нет явного упоминания продукта или услуги, не включай его.
-   **Символы `****`:** Это либо конфиденциальные данные (номера телефонов), либо ненормативная лексика. Учитывай общий контекст вокруг них для определения тональности.
-   **Если темы нет:** Если в отзыве невозможно определить ни одну тему, верни пустой массив `topic_sentiment_pairs`.

Вот список всех продуктов банка на сегодняшний день:
* **дебетовые карты:**
    * Умная дебетовая карта «Мир»
    * Премиальная карта Mir Supreme
    * Дебетовая карта с кэшбэком для самозанятых
    * Умная дебетовая карта «Мир»
    * Карта для автолюбителей «Газпромбанк—Газпромнефть»
    * Виртуальная дебетовая карта ГПБ&ФК «Зенит»
    * Дебетовая Пенсионная карта
* **Кредитные карты:**
    * Кредитная карта с льготным периодом до 120 дней
    * Простая кредитная карта
    * Кредитная карта 90 дней
    * Кредитная карта 180 дней Премиум
    * Кредитная карта для самозанятых
* **Накопительные счета:**
    * Накопительный счет
    * «Ежедневная выгода»
    * «Ежедневный процент»
    * «Премиум»
    * Социальный счет
* **Вклады:**
    * Вклад «Новые деньги»
    * Вклад «Ключевой момент»
    * Вклад «Копить»
    * Вклад «В Плюсе»
    * Вклад «Расширяй возможности»
    * Социальный вклад
* **Кредиты:**
    * Кредит наличными
    * Кредит наличными под залог недвижимости
    * Кредит на авто и другие цели
    * Рефинансирование потребительских кредитов
    * Дачный кредит
    * Кредит на образование
    * Кредит наличными для бюджетников
* **Другие услуги банка:**
    * Газпромбанк Мобайл
    * Газпромбанк Travel
    * Gazprom Pay
    * GorodPay
    * Газпром Бонус
    * Страховые и сервисные продукты
    * Депозитарные услуги


**Формат ответа — строго JSON:**
{
    "topic_sentiment_pairs": [
        {
            "topic": "Название темы 1",
            "sentiment": "positive/negative/neutral"
        },
        {
            "topic": "Название темы 2",
            "sentiment": "positive/negative/neutral"
        }
    ]
}


**Примеры:**
Отзыв 1: Мобильное приложение просто ужасное, постоянно вылетает. А вот вклад оформил недавно — условия хорошие.

Ответ:
{
    "topic_sentiment_pairs": [
        {
            "topic": "Мобильное приложение",
            "sentiment": "negative"
        },
        {
            "topic": "Вклады",
            "sentiment": "positive"
        }
    ]
}


Отзыв 2: Звонил узнать насчет ипотеки, но в поддержке ничем не помогли.

Ответ:
{
    "topic_sentiment_pairs": [
        {
            "topic": "Дистанционное обслуживание",
            "sentiment": "negative"
        },
        {
            "topic": "Ипотека",
            "sentiment": "neutral"
        }
    ]
},

**Вот отзывы клиентов, которые необходимо обработать:**
"""

In [3]:
import json
import pandas as pd
import numpy as np

import glob
import os

In [4]:
df = pd.read_csv("data/data_latest.csv")

df

,review_id,date,review_text,topic,subtopic,sentiment
0,1000087,2025-09-19,Вклад «Новые деньги» невозможно оформить без п...,Вклады,NaN,Negative
1,999494,2025-09-18,В июне 2025 года я порекомендовал премиальную ...,Дебетовые карты,NaN,Negative
2,999142,2025-09-17,Мошенниччиские аперации в интересах Ренессанс ...,Обслуживание,NaN,Negative
3,998360,2025-09-15,Купил услугу Газпром Бонус «Премиум» за 2 990 ...,Дебетовые карты,NaN,Negative
4,998516,2025-09-15,Производил оформление открытия срочного банков...,Вклады,«Накопительный»,Negative
...,...,...,...,...,...,...
4773,7470,2011-04-07,Ужастное обслуживание! Мало того потеряли доку...,Обслуживание,NaN,Negative
4774,7049,2011-03-28,Могут заблокировать рассчетную или кредитную к...,Кредитные карты,NaN,Negative
4775,5221,2011-01-25,"Мало того уже прошла неделя, а ПТС так и не ве...",Автокредиты,NaN,Negative
4776,5053,2011-01-16,Газпромбанк– отличный банк с отличными сотрудн...,Ипотека,NaN,Positive


In [5]:
print(df["review_text"].values[8])

Прошлым летом оформил дебетовую карту Юнион пей от Газпромбанка, которую обещали выпустить бесплатно при соблюдении условий. Условия я выполнил, но списывались платы за подписки, на которые я не подписывался и в итоге 5000 руб. Не вернули за выпуск карты, которая к слову не работает за границей и нигде ей не воспользовался. Но разговор не об этом, вместе с дебетовой курьер привез мне кредитную карту, я сказал, что ее не заказывал, но он сказал они идут вместе, но мой вопрос нужно ли что-то за нее платить, был дан четкий вопрос, что нет. Однако через месяц случайно обнаружил задолженность порядка 4600 рублей, хотя карта не активирована. Выяснилось, что это было начислено за навязанные какие-то страховки и еще что-то. Я составил через чат обращение, где описал ситуацию, два раза попросил оператора закрыть карту. Заявка была рассмотрена, деньги вернулись. Спустя год с лишним обнаруживаю задолженность более 3000 руб за второй год обслуживания счета, который я думал закрыт, так как составля

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://192.168.0.193:8100/v1", api_key="api_key")

response = client.chat.completions.create(
    model="JosephThePatrician/gemma3-270m-it-reviews-v1",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": df["review_text"].values[8].replace("\n", " ")}
    ],
    temperature=0.1
)

In [7]:
response

ChatCompletion(id='chatcmpl-99c089ffcca54c5ab8cfd79e14e66b0a', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="[{'topic': 'Дебетовые карты', 'sentiment': 'negative'}, {'topic': 'Кредитные карты', 'sentiment': 'negative'}, {'topic': 'Страховые и сервисные продукты', 'sentiment': 'negative'}]", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning_content=None), stop_reason=None)], created=1759086302, model='JosephThePatrician/gemma3-270m-it-reviews-v1', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=51, prompt_tokens=1606, total_tokens=1657, completion_tokens_details=None, prompt_tokens_details=None), prompt_logprobs=None, kv_transfer_params=None)

In [53]:
response.choices[0].message.content

"[{'topic': 'Дебетовые карты', 'sentiment': 'negative'}, {'topic': 'Кредитные карты', 'sentiment': 'neutral'}, {'topic': 'Дистанционное обслуживание', 'sentiment': 'negative'}]"